# Solution Key — Object-Oriented Programming
## All Five Labs

These are *reference* solutions — most labs say "how you define it is up to you," so reward any correct, well-reasoned implementation. Each lab includes a verified sample run. The calculator solution also ships as **`calculator.py`** (the notebook imports it).

---
## Lab 1 — Augment `BankAccount` (+ `Calculator`)

Add `__eq__`, `__mul__`, and `__len__` to the account class, and build a printing `Calculator`.

In [ ]:
class BankAccount:
    def __init__(self, name, balance):
        self.name = name
        self.balance = balance

    def __repr__(self):
        return f'{self.__class__.__name__}({self.name!r}, {self.balance!r})'

    def __str__(self):
        return f'{self.name} has {self.balance} in the bank'

    # --- Lab additions ---
    def __eq__(self, other):
        # Design choice: two accounts are 'equal' if their balances match.
        return self.balance == other.balance

    def __mul__(self, factor):
        # Create a NEW account; tag the name and scale the balance.
        return BankAccount(f'{self.name} x{factor}', self.balance * factor)

    def __len__(self):
        # __len__ must return a non-negative int; length of the owner's name is a fine choice.
        return len(self.name)

    def deposit(self, amount):
        if amount > 0:
            self.balance += amount
            return self.balance
        print("can't deposit nonpositive amount!")

    def withdraw(self, amount):
        if amount > 0 and amount <= self.balance:
            self.balance -= amount
            return self.balance
        print("can't withdraw", amount, 'or you would be overdrawn!')

**Sample run:**

In [ ]:
a = BankAccount('Ada', 100)
b = BankAccount('Bo', 100)
c = BankAccount('Cy', 50)
print('a == b:', a == b, '| a == c:', a == c)
print('a * 3 :', repr(a * 3))
print('len(a):', len(a))

The calculator solution (also in `calculator.py`). One arg operates on the running total; two args operate on the operands and replace it. Every step is recorded for `showcalc()`; `ac()` clears.

In [ ]:
class Calc:
    """Running-total printing calculator."""
    def __init__(self):
        self.total = 0
        self.history = []

    def _step(self, a, op, b, result):
        self.history.append(f'{a} {op} {b} = {result}')
        self.total = result
        return result

    def add(self, a, b=None):
        return self._step(self.total, '+', a, self.total + a) if b is None \
            else self._step(a, '+', b, a + b)

    def sub(self, a, b=None):
        return self._step(self.total, '-', a, self.total - a) if b is None \
            else self._step(a, '-', b, a - b)

    def mult(self, a, b=None):
        return self._step(self.total, '*', a, self.total * a) if b is None \
            else self._step(a, '*', b, a * b)

    def div(self, a, b=None):
        return self._step(self.total, '/', a, self.total / a) if b is None \
            else self._step(a, '/', b, a / b)

    def pow(self, a, b=None):
        return self._step(self.total, '**', a, self.total ** a) if b is None \
            else self._step(a, '**', b, a ** b)

    def log(self, a):
        from math import log as _log
        self.history.append(f'log({a}) = {_log(a)}')
        self.total = _log(a)
        return self.total

    def showcalc(self):
        return '\n'.join(self.history)

    def __str__(self):
        return self.showcalc()

    def ac(self):
        self.total = 0
        self.history = []

**Sample run (matches the notebook's demo cells):**

In [ ]:
c = Calc()
c.add(2, 5)
c.add(9)
c.mult(13)
c.mult(20, 5)
print(c)
c.ac()
print('after ac(), add(1) ->', c.add(1))

> **Note:** `__len__` must return a non-negative `int` (a common bug is returning the balance as a float). For `Calc`, the key behaviors are the 1-arg-vs-2-arg distinction, that `showcalc()` reconstructs every step, and that `ac()` truly resets both the total and the history.

---
## Lab 2 — `FunnyList`

A `list` where order doesn't matter for equality: `[1, 2, 3] == [3, 1, 2]`.

In [ ]:
class FunnyList(list):
    """A list whose equality ignores order."""
    def __eq__(self, other):
        return sorted(self) == sorted(other)

    def __ne__(self, other):
        return not self.__eq__(other)

    __hash__ = None   # defining __eq__ makes instances unhashable (lists already are)

**Sample run:**

In [ ]:
print(FunnyList([1, 2, 3]) == [3, 1, 2])   # True  - same items, any order
print(FunnyList([1, 2, 3]) == [1, 2, 4])   # False - different items
print(FunnyList([1, 2, 3]) == [1, 2, 3, 3]) # False - different length

> **Note:** `sorted()` comparison is the clean approach for orderable items. A student who compares `collections.Counter(self) == Counter(other)` is arguably *better* (handles duplicates and unorderable-but-hashable items) — accept it. Watch for forgetting that overriding `__eq__` alone leaves `!=` working via Python's default negation, so `__ne__` is optional on 3.x but harmless to include.

---
## Lab 3 — `ZanyInt`

Inherit from `int`: make `len()` work, give `str()` odd behavior (`str(3)` → `'three'`, others padded with spaces), and make `+` *usually* right but occasionally wrong (via `random`).

In [ ]:
import random


class ZanyInt(int):
    _WORDS = {3: 'three'}   # extend as you like

    def __len__(self):
        # ints have no len(); define it as the digit count.
        return len(str(abs(int(self))))

    def __str__(self):
        if int(self) in self._WORDS:
            return self._WORDS[int(self)]
        return f'  {int(self)}  '          # leading/trailing spaces

    def __add__(self, other):
        result = int(self) + int(other)
        if random.random() < 0.25:          # ~25% of the time, be 'zany'
            result += 1
        return ZanyInt(result)

**Sample run** (seeded so the 'zany' additions are reproducible here):

In [ ]:
random.seed(42)
print('len(ZanyInt(12345)) =', len(ZanyInt(12345)))
print('str(ZanyInt(3))     =', repr(str(ZanyInt(3))))
print('str(ZanyInt(7))     =', repr(str(ZanyInt(7))))
print('2 + 2 eight times   =', [int(ZanyInt(2) + 2) for _ in range(8)])

> **Note:** `__len__` must return a non-negative int. The `+` override must still return a `ZanyInt` (so chained operations stay zany) — returning a plain `int` is the common miss. Using `random.random() < p` is the cleanest way to make it 'sometimes' wrong; any reasonable use of `random` is fine.

---
## Lab 4 — Tracking Instance Names (class vs instance variables)

Use a **class variable** to record names of all objects ever created, and a second one (with `__del__`) to track only those currently alive.

In [ ]:
class TrackedPerson:
    all_names = []        # class variable: every name ever created
    current_names = []    # class variable: names currently alive

    def __init__(self, name):
        self.name = name              # instance variable
        TrackedPerson.all_names.append(name)
        TrackedPerson.current_names.append(name)

    def __del__(self):
        # Called when the instance is garbage-collected (refcount hits 0).
        TrackedPerson.current_names.remove(self.name)

**Sample run:**

In [ ]:
a = TrackedPerson('Anu')
b = TrackedPerson('Bea')
c = TrackedPerson('Cai')
del b                      # Bea is gone
print('ever created:', TrackedPerson.all_names)
print('still alive  :', TrackedPerson.current_names)

> **Note:** the class variables must be referenced as `TrackedPerson.all_names` (not `self.all_names = ...`, which would shadow the class var with an instance var). `__del__` timing depends on garbage collection — fine for an explicit `del`, but worth mentioning it isn't guaranteed to fire promptly in every situation (e.g., reference cycles).

---
## Lab 5 — Value with a Modification Counter (`__setattr__`)

Hold a `value` and a `counter` that increments each time `value` changes. Other attributes don't bump the counter; direct writes to `counter` are rejected. `super().__setattr__` does the real work (assigning to `self.x` normally would re-enter `__setattr__` forever).

In [ ]:
class ModTracker:
    def __init__(self, value=None):
        # Use super() so these initial sets don't recurse and don't over-count.
        super().__setattr__('counter', 0)
        super().__setattr__('value', value)

    def __setattr__(self, name, val):
        if name == 'counter':
            raise AttributeError("can't modify the counter directly")
        if name == 'value':
            super().__setattr__('counter', self.counter + 1)
        super().__setattr__(name, val)

**Sample run:**

In [ ]:
m = ModTracker('start')
m.value = 'a'
m.value = 'b'
m.label = 'note'          # not 'value' -> counter unchanged
print('value:', m.value, '| counter:', m.counter)
try:
    m.counter = 99
except AttributeError as e:
    print('rejected:', e)

> **The 'why super()?' answer** (the lab asks): inside `__setattr__`, writing `self.value = val` would call `__setattr__` again — infinite recursion. `super().__setattr__(...)` reaches the base `object` implementation that actually stores the attribute. 
>
> **Note:** counter increments only on `value` changes (here it ends at 2), other attributes are allowed without bumping it, and direct `counter` writes are blocked.